# Kafka → Bronze Ingestion

This notebook ingests retail transaction events from Aiven Kafka using Spark Structured Streaming and persists them as a Delta table in the Bronze layer.

**Source**
- Kafka service: Aiven Kafka
- Topic: `retail-transactions`

**Target**
- Catalog: `retailanalytics`
- Schema: `bronze`
- Table: `transaction_raw`

**Processing**
- Kafka authentication: SASL/SCRAM
- Transport security: SASL_SSL
- Transaction payload: JSON
- Storage format: Delta

In [0]:
# Kafka configuration

kafka_bootstrap_servers = (
    "retail-streaming-kafka-retail-data-platform.g.aivencloud.com:21758"
)

kafka_topic = "retail-transactions"

# Retrieve Kafka credentials from Unity Catalog Secrets
kafka_username = dbutils.secrets.get(
    "retailanalytics",
    "secrets",
    "kafka_username"
)

kafka_password = dbutils.secrets.get(
    "retailanalytics",
    "secrets",
    "kafka_password"
)

# Kafka SSL truststore
truststore_path = (
    "/Volumes/retailanalytics/secrets/"
    "kafkacerts/aiven-truststore.jks"
)

truststore_password = "changeit"

# Bronze target
bronze_table = "retailanalytics.bronze.transaction_raw"

bronze_checkpoint = (
    "/Volumes/retailanalytics/secrets/"
    "kafkacerts/checkpoints/bronze_transaction_raw"
)

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    TimestampType
)

transaction_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("transaction_ts", TimestampType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("store_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("gross_amount", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("transaction_status", StringType(), True)
])

In [0]:
kafka_stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
    .option("subscribe", kafka_topic)
    .option("startingOffsets", "earliest")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "SCRAM-SHA-256")
    .option(
        "kafka.sasl.jaas.config",
        f'kafkashaded.org.apache.kafka.common.security.scram.ScramLoginModule required '
        f'username="{kafka_username}" password="{kafka_password}";'
    )
    .option(
        "kafka.ssl.truststore.location",
        truststore_path
    )
    .option(
        "kafka.ssl.truststore.password",
        truststore_password
    )
    .option(
        "kafka.ssl.truststore.type",
        "JKS"
    )
    .load()
)

In [0]:
from pyspark.sql.functions import from_json, col

bronze_stream_df = (
    kafka_stream_df
    .select(
        from_json(
            col("value").cast("string"),
            transaction_schema
        ).alias("transaction"),
        col("topic"),
        col("partition"),
        col("offset"),
        col("timestamp").alias("kafka_timestamp")
    )
    .select(
        "transaction.*",
        "topic",
        "partition",
        "offset",
        "kafka_timestamp"
    )
)

In [0]:
bronze_query = (
    bronze_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", bronze_checkpoint)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

bronze_query.awaitTermination()

In [0]:
%sql
SELECT COUNT(*) AS transaction_count
FROM retailanalytics.bronze.transaction_raw;